In [1]:
!pip install annoy faiss-cpu hnswlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.5/647.5 kB 5.3 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 14.0 MB/s  0:00:01
  Created wheel for annoy: filename=annoy-1.17.3-cp313-cp313-linux_x86_64.whl size=520036 sha256=0008e4743279d634ff3f42ed0ef5ab8f010bbb8c01808680275c48a55a67539e
  Stored in directory: /root/.cache/pip/wheels/fc/4e/87/f1f957e7382aa370452cff98276f51686924d5415e449800ed
  Created wheel for hnswlib: filename=hnswlib-0.8.0-cp313-cp313-linux_x86_64.whl size=2679801 sha256=d41bb706801d835956519cc9677d70cb8af02efa146934793c0a36285fc226f1
  Stored in directory: /root/.cache/pip/wheels/35/04/88/b31765a4b9957705e18065db4657e61fc8da54f50

In [6]:
import os
import kagglehub
import pandas as pd
import numpy as np
import time
import faiss
from annoy import AnnoyIndex
import hnswlib
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

path = kagglehub.dataset_download("bwandowando/spotify-songs-with-attributes-and-lyrics")
print("Path to dataset files:", path)

csv_files = sorted([f for f in os.listdir(path) if f.endswith('.csv')])
print("Daftar file CSV di dataset:", csv_files)

first_csv_path = os.path.join(path, csv_files[0])
print("File CSV pertama yang dipakai:", first_csv_path)

df = pd.read_csv(first_csv_path)

features = [
    'danceability', 'energy', 'loudness', 'speechiness',
    'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo'
]

df = df.dropna(subset=features).reset_index(drop=True)

X = df[features].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X).astype(np.float32)  # float32 wajib untuk FAISS & HNSW

n_data, dim = X_scaled.shape
k = 10

query_idx = 0
query = X_scaled[query_idx:query_idx+1]

print(f"Total data lagu: {n_data} | Dimensi fitur: {dim}")
if 'track_name' in df.columns and 'track_artist' in df.columns:
    print(f"Lagu Query: {df.loc[query_idx, 'track_name']} - {df.loc[query_idx, 'track_artist']}")

100%|██████████| 894M/894M [00:40<00:00, 23.0MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/bwandowando/spotify-songs-with-attributes-and-lyrics/versions/20
Daftar file CSV di dataset: ['songs_with_attributes_and_lyrics.csv', 'songs_with_lyrics_and_timestamps.csv']
File CSV pertama yang dipakai: /root/.cache/kagglehub/datasets/bwandowando/spotify-songs-with-attributes-and-lyrics/versions/20/songs_with_attributes_and_lyrics.csv
Total data lagu: 955320 | Dimensi fitur: 9


In [7]:
#3 - Exact NN (Baseline)
nn = NearestNeighbors(n_neighbors=k, algorithm='brute', metric='euclidean')

start = time.time()
nn.fit(X_scaled)
exact_build = time.time() - start

start = time.time()
dist_exact, idx_exact = nn.kneighbors(query)
exact_query = time.time() - start

exact_indices = idx_exact[0]

print("=== Exact NN ===")
print("Indices:", exact_indices)
print(f"Build Time: {exact_build:.4f} s | Query Time: {exact_query*1000:.4f} ms")

=== Exact NN ===
Indices: [     0 394553 764272 837727 749223  61511 625227 833164 676733 523698]
Build Time: 0.0229 s | Query Time: 95.7730 ms


In [8]:
#4 - ANNOY
ann_index = AnnoyIndex(dim, 'euclidean')

start = time.time()
for i in range(n_data):
    ann_index.add_item(i, X_scaled[i])
ann_index.build(10)  # 10 trees
ann_build = time.time() - start

start = time.time()
ann_indices = ann_index.get_nns_by_vector(query[0], k)
ann_query = time.time() - start

ann_recall = len(set(ann_indices) & set(exact_indices)) / k

print("=== ANNOY ===")
print("Indices:", ann_indices)
print(f"Build Time: {ann_build:.4f} s | Query Time: {ann_query*1000:.4f} ms | Recall: {ann_recall:.2f}")

=== ANNOY ===
Indices: [0, 764272, 749223, 625227, 833164, 676733, 523698, 85956, 888081, 763625]
Build Time: 17.3866 s | Query Time: 0.2334 ms | Recall: 0.70


In [11]:
#5 - FAISS
nlist = min(50, max(1, n_data // 40))
m = 3
quantizer = faiss.IndexFlatL2(dim)
faiss_index = faiss.IndexIVFPQ(quantizer, dim, nlist, m, 8)

start = time.time()
faiss_index.train(X_scaled)
faiss_index.add(X_scaled)
faiss_build = time.time() - start

faiss_index.nprobe = min(10, nlist)
start = time.time()
dist_faiss, idx_faiss = faiss_index.search(query, k)
faiss_query = time.time() - start

faiss_indices = idx_faiss[0]
faiss_recall = len(set(faiss_indices) & set(exact_indices)) / k

print("=== FAISS (IVF+PQ) ===")
print("Indices:", faiss_indices)
print(f"Build Time: {faiss_build:.4f} s | Query Time: {faiss_query*1000:.4f} ms | Recall: {faiss_recall:.2f}")

=== FAISS (IVF+PQ) ===
Indices: [     0 546864  85956 466594 466595 749223 574236 113181 770827 833164]
Build Time: 8.5892 s | Query Time: 13.2740 ms | Recall: 0.30


In [12]:
#6 - HNSW
hnsw_index = hnswlib.Index(space='l2', dim=dim)

start = time.time()
hnsw_index.init_index(max_elements=n_data, ef_construction=200, M=16)
hnsw_index.add_items(X_scaled)
hnsw_build = time.time() - start

hnsw_index.set_ef(50)
start = time.time()
idx_hnsw, dist_hnsw = hnsw_index.knn_query(query, k=k)
hnsw_query = time.time() - start

hnsw_indices = idx_hnsw[0]
hnsw_recall = len(set(hnsw_indices) & set(exact_indices)) / k

print("=== HNSW ===")
print("Indices:", hnsw_indices)
print(f"Build Time: {hnsw_build:.4f} s | Query Time: {hnsw_query*1000:.4f} ms | Recall: {hnsw_recall:.2f}")

=== HNSW ===
Indices: [     0 394553 764272 837727 749223  61511 625227 833164 676733 523698]
Build Time: 203.8767 s | Query Time: 0.4628 ms | Recall: 1.00


In [13]:
#7 - Perbandingan Performa
summary_df = pd.DataFrame([
    {"Algoritma": "Exact NN", "Build Time (s)": round(exact_build, 4), "Query Time (ms)": round(exact_query*1000, 4), "Recall@10": 1.00},
    {"Algoritma": "ANNOY",    "Build Time (s)": round(ann_build, 4),   "Query Time (ms)": round(ann_query*1000, 4),   "Recall@10": ann_recall},
    {"Algoritma": "FAISS",    "Build Time (s)": round(faiss_build, 4), "Query Time (ms)": round(faiss_query*1000, 4), "Recall@10": faiss_recall},
    {"Algoritma": "HNSW",     "Build Time (s)": round(hnsw_build, 4),  "Query Time (ms)": round(hnsw_query*1000, 4),  "Recall@10": hnsw_recall},
])
print("=== Perbandingan Performa ===")
display(summary_df)

if 'track_name' in df.columns and 'track_artist' in df.columns:
    tracks_comparison = pd.DataFrame({
        "Rank": range(1, k + 1),
        "Exact NN": df.loc[exact_indices, 'track_name'].values + " - " + df.loc[exact_indices, 'track_artist'].values,
        "ANNOY":    df.loc[ann_indices, 'track_name'].values   + " - " + df.loc[ann_indices, 'track_artist'].values,
        "FAISS":    df.loc[faiss_indices, 'track_name'].values + " - " + df.loc[faiss_indices, 'track_artist'].values,
        "HNSW":     df.loc[hnsw_indices, 'track_name'].values  + " - " + df.loc[hnsw_indices, 'track_artist'].values,
    })
    print("\n=== Perbandingan 10 Lagu Terdekat ===")
    display(tracks_comparison)

=== Perbandingan Performa ===


,Algoritma,Build Time (s),Query Time (ms),Recall@10
0,Exact NN,0.0229,95.7730,1.0
1,ANNOY,17.3866,0.2334,0.7
2,FAISS,8.5892,13.2740,0.3
3,HNSW,203.8767,0.4628,1.0
